# 08. Annual model maps: Paoay and the Atok boundary

Applies the frozen annual model (the current Training 04 model) to every study pixel of the best-covered year. Pixels without enough clear half-months are "No features"; predictions below 0.60 top probability or 0.10 margin are "Unknown crop type". The map also flags pixels with more than 20% of predictors outside the training 5th–95th percentile range. Paoay uses the land-activity cropland map; the Atok boundary KMZ uses ESA WorldCover 2021 cropland.

**Needs:** the Paoay KMZ files, `atok_imagery_root` composites and the land-activity map from Paoay 06, and Sentinel-1 and WorldCover grids for the chosen year (fetched once from Earth Engine and cached under `pipeline_outputs/annual/cache/`).

These maps are preliminary: no dated local labels exist yet, so they carry no accuracy estimate. Each pixel is 10 m × 10 m, so hectares are pixel counts ÷ 100.

In [ ]:
from pathlib import Path
import json
import sys

root = next(path for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
            if (path / "pyproject.toml").is_file())
if str(root / "src") not in sys.path:
    sys.path.insert(0, str(root / "src"))

from crop_pipeline.training import annual_maps

In [ ]:
for area in ("paoay", "atok"):
    receipt = annual_maps.map_area(root, area)
    summary = json.loads((receipt.parent / "map_summary.json").read_text())
    print(area, receipt.relative_to(root))
    print({key: summary[key] for key in ("year", "share_with_features", "confident_share_in_cropland",
                                         "outside_training_range_share", "crop_shares_of_confident_cropland")})

## Hectares by area

Crop, unknown and no-feature hectares for the whole boundary, cropland and each reference polygon.

In [ ]:
import csv

from crop_pipeline.analysis_workflow import resolve_stage_pointer

columns = {"carrot": "carrot", "cabbage": "cabbage", "potato": "potato", "Unknown crop type": "unknown",
           "No features": "no features"}
for area in ("paoay", "atok"):
    run = resolve_stage_pointer(root / f"pipeline_outputs/annual/current_{area}.json").parent
    print(f"\n{area} ({run.name}), hectares")
    print(f"{'area':22} {'total':>8} " + " ".join(f"{name:>11}" for name in columns.values()))
    with (run / "area_summary.csv").open() as stream:
        for row in csv.DictReader(stream):
            print(f"{row['area'][:22]:22} {int(row['pixels']) / 100:8.1f} "
                  + " ".join(f"{int(row[key]) / 100:11.1f}" for key in columns))

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
import numpy as np
import rasterio

palette = ListedColormap(["#fcfcfb", "#eb6834", "#1baf7a", "#2a78d6", "#c9c8c3", "#efeeea"])
fig, axes = plt.subplots(1, 2, figsize=(12, 6))
for ax, area in zip(axes, ("paoay", "atok")):
    run = resolve_stage_pointer(root / f"pipeline_outputs/annual/current_{area}.json").parent
    with rasterio.open(run / "crop_map.tif") as source:
        code, cropland = source.read(1), source.read(2)
    ax.imshow(np.where(cropland == 2, code, np.where(code > 0, 5, 0)), cmap=palette, vmin=-.5, vmax=5.5,
              interpolation="nearest")
    ax.set_title(f"{area.capitalize()}, cropland only (preliminary, unvalidated)", loc="left", fontsize=10)
    ax.set_axis_off()
fig.legend(handles=[Patch(color=c, label=l) for c, l in (("#eb6834", "Carrot"), ("#1baf7a", "Cabbage"),
                                                         ("#2a78d6", "Potato"), ("#c9c8c3", "Unknown"),
                                                         ("#efeeea", "Not cropland or no features"))],
           loc="lower center", ncol=5, frameon=False)
plt.show()